# 6. Symmetric and Hermitian eigensolvers

BatchLAS ships several full-spectrum symmetric eigensolvers. They all compute
the same thing — they differ in how the work is mapped onto the device.

| Variant | Approach | Best for |
|---|---|---|
| `syev` | general driver, vendor path where available | anything |
| `syev_cta` | `sytrd_cta` → `steqr_cta` → `ormqx_cta`, one work-group per matrix | $n \le 32$ |
| `syev_jacobi_cta` | two-sided Jacobi in one work-group | $n \le 32$, graded input |
| `syev_blocked` | blocked reduction + divide and conquer | medium/large $n$ |
| `syev_two_stage` | dense → band → tridiagonal reduction | large $n$ |

`syev_variant_support()` asks the device which of these it can actually run,
instead of guessing from the matrix size.

Notebook **10** shows why `syev_jacobi_cta` is worth having; notebook **12**
measures which variant is fastest where.

In [1]:
import numpy as np

import batchlas as bl

from _common import (
    batched_symmetric,
    eigenvalue_error,
    header,
    preferred_device,
    report,
    residual,
    section,
)

header("6. Symmetric eigensolvers")

device = preferred_device()
batch = 8


6. Symmetric eigensolvers


## Which variants does this device support?

In [2]:
section("Which variants does this device support?")

small = batched_symmetric(batch, 16, seed=1)
for key, value in bl.syev_variant_support(small, uplo="lower", device=device).items():
    report(key, value)


-- Which variants does this device support?
          device: NVIDIA GeForce RTX 4090
          is_gpu: True
          max_sub_group: 32
          cta: True
          blocked: True
          two_stage: True


## `syev` — eigenvalues and eigenvectors

Returns `(w, V)` with eigenvalues ascending. The input is read from the lower
triangle by default. The residual $\lVert A V - V \, \mathrm{diag}(w) \rVert$
is the check that matters.

In [3]:
section("syev: eigenvalues and eigenvectors")

reference = np.linalg.eigvalsh(small)
values, vectors = bl.syev(small, device=device)

report("eigenvalue error", eigenvalue_error(values, reference), tol=1e-10)
report("|A V - V diag(w)|", residual(small, values, vectors), tol=1e-10)


-- syev: eigenvalues and eigenvectors
   [ok  ] eigenvalue error: 1.954e-14  (tol 1.0e-10)
   [ok  ] |A V - V diag(w)|: 8.882e-15  (tol 1.0e-10)


### Eigenvalues only

Pass `compute_vectors=False` to skip the back-transform entirely; the call then
returns just the eigenvalue array.

In [4]:
section("Eigenvalues only")

values = bl.syev(small, compute_vectors=False, device=device)
report("shape", values.shape)
report("error", eigenvalue_error(values, reference), tol=1e-10)


-- Eigenvalues only
          shape: (8, 16)
   [ok  ] error: 1.954e-14  (tol 1.0e-10)


## The small-matrix variants

Both keep one matrix resident in a single work-group. They need a GPU with a
sub-group width of 32 and are limited to $n \le 32$.

In [5]:
section("The small-matrix variants (n <= 32): one work-group per problem")

for name in ("syev_cta", "syev_jacobi_cta"):
    values, vectors = getattr(bl, name)(small, device=device)
    report(f"{name:16s} eigenvalue error", eigenvalue_error(values, reference), tol=1e-10)
    report(f"{name:16s} residual", residual(small, values, vectors), tol=1e-10)


-- The small-matrix variants (n <= 32): one work-group per problem
   [ok  ] syev_cta         eigenvalue error: 1.954e-14  (tol 1.0e-10)
   [ok  ] syev_cta         residual: 8.882e-15  (tol 1.0e-10)
   [ok  ] syev_jacobi_cta  eigenvalue error: 3.286e-14  (tol 1.0e-10)
   [ok  ] syev_jacobi_cta  residual: 1.688e-14  (tol 1.0e-10)


## The medium and large variants

In [6]:
section("The medium/large variants")

medium = batched_symmetric(4, 128, seed=2)
medium_reference = np.linalg.eigvalsh(medium)

for name in ("syev", "syev_blocked", "syev_two_stage"):
    try:
        values, vectors = getattr(bl, name)(medium, device=device)
        report(f"{name:16s} eigenvalue error", eigenvalue_error(values, medium_reference), tol=1e-8)
        report(f"{name:16s} residual", residual(medium, values, vectors), tol=1e-8)
    except (RuntimeError, NotImplementedError) as exc:
        report(f"{name:16s}", f"unavailable ({type(exc).__name__}: {exc})")


-- The medium/large variants
   [ok  ] syev             eigenvalue error: 6.573e-14  (tol 1.0e-08)
   [ok  ] syev             residual: 1.066e-14  (tol 1.0e-08)
   [ok  ] syev_blocked     eigenvalue error: 6.573e-14  (tol 1.0e-08)
   [ok  ] syev_blocked     residual: 1.066e-14  (tol 1.0e-08)


   [ok  ] syev_two_stage   eigenvalue error: 6.573e-14  (tol 1.0e-08)
   [ok  ] syev_two_stage   residual: 1.199e-14  (tol 1.0e-08)


## Tuning a variant through its options object

Each variant is driven by a different inner algorithm, and each has its own
options dataclass:

- `syev_cta` → `SteqrOptions` (a CTA-resident QR iteration)
- `syev_blocked` / `syev_two_stage` → `StedcOptions` (divide and conquer)
- `syev_jacobi_cta` → `JacobiOptions`

A plain `dict` works anywhere an options object does.

In [7]:
section("Tuning a variant through its options object")

options = bl.SteqrOptions(max_sweeps=200, cta_shift_strategy="wilkinson", sort_order="ascending")
values, _ = bl.syev_cta(small, options=options, device=device)
report("eigenvalue error", eigenvalue_error(values, reference), tol=1e-10)

options = bl.StedcOptions(recursion_threshold=32, max_sec_iter=80)
try:
    values, _ = bl.syev_blocked(medium, options=options, device=device)
    report("blocked with custom stedc options", eigenvalue_error(values, medium_reference), tol=1e-8)
except (RuntimeError, NotImplementedError) as exc:
    report("blocked with custom stedc options", f"unavailable ({type(exc).__name__})")


-- Tuning a variant through its options object
   [ok  ] eigenvalue error: 1.954e-14  (tol 1.0e-10)


   [ok  ] blocked with custom stedc options: 6.395e-14  (tol 1.0e-08)


## Hermitian (complex) input

Complex input works through the same calls; the eigenvalues come back real.

In [8]:
section("Hermitian (complex) input")

rng = np.random.default_rng(3)
z = rng.standard_normal((batch, 16, 16)) + 1j * rng.standard_normal((batch, 16, 16))
z = (z + z.conj().transpose(0, 2, 1)) / 2.0
z_reference = np.linalg.eigvalsh(z)

values, vectors = bl.syev(z, device=device)
report("eigenvalue error", eigenvalue_error(values, z_reference), tol=1e-10)
report("|A V - V diag(w)|", residual(z, values.astype(z.dtype), vectors), tol=1e-10)


-- Hermitian (complex) input
   [ok  ] eigenvalue error: 1.776e-14  (tol 1.0e-10)
   [ok  ] |A V - V diag(w)|: 5.784e-15  (tol 1.0e-10)


## `uplo` — which triangle holds your data

Both triangles of a full symmetric matrix give the same spectrum. Supplying
only the lower triangle also works, which confirms the strict upper half really
is ignored for `uplo="lower"`.

> The mirror image — upper triangle only with `uplo="upper"` — is **not**
> reliable on the CUDA path today. See the README. Pass the full matrix instead.

In [9]:
section("uplo: which triangle holds your data")

for side in ("lower", "upper"):
    values = bl.syev(small, compute_vectors=False, uplo=side, device=device)
    report(f"uplo={side}", eigenvalue_error(values, reference), tol=1e-10)

values = bl.syev(np.tril(small), compute_vectors=False, uplo="lower", device=device)
report("lower triangle only", eigenvalue_error(values, reference), tol=1e-10)


-- uplo: which triangle holds your data
   [ok  ] uplo=lower: 1.954e-14  (tol 1.0e-10)
   [ok  ] uplo=upper: 1.954e-14  (tol 1.0e-10)
   [ok  ] lower triangle only: 1.954e-14  (tol 1.0e-10)
